# RNA-seq・Gene Expression analysis

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/rnaseq_en.png?raw=true" alt="central_dogma" height="280px" align="middle">

In this lecture, we will focus on transcriptome analysis using RNA-seq, a technique used to examine when genes are expressed and at what levels.


# What is RNA-seq

From here, we will learn about RNA-seq, a sequencing technology that targets RNA.

After a gene is transcripted into mRNA, it is translated into an amino acid sequence.

The amino acid sequence then folds into a three-dimensional structure to form a protein, which carries out specific functions.

This process is known as the central dogma.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/central_dogma.png?raw=true" alt="central_dogma" height="280px" align="middle">

DNA sequencing enable us to read DNA sequences using long-read or short-read sequencing technologies.

In contrast, RNA sequencing (RNA-seq) reads transcripted mRNA.

Using RNA-seq, we can obtain information on which genes are transcripted and how much they are transcripted in a given sample (**gene expression levels**).

For example, by comparing RNA-seq data from samples with and without a specific treatment, we can identify which genes are expressed in response to that treatment.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/RNAseq_compare.png?raw=true" alt="central_dogma" height="350px" align="middle">

# Mapping of RNA-seq

For RNA-seq analysis, we begin with mapping.

First, mRNA from expressed genes is extracted and sequenced as short reads.

Since which gene each read comes from is not known beforehand, mapping is used to determine which genes were expressed.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/mRNA_mapping.png?raw=true" alt="central_dogma" height="350px" align="middle">

However, mapping RNA-seq differs from mapping DNA sequences.

As a review of basic biology, during transcription into mRNA, unnecessary regions (introns) are removed, and only exons are joined together through a process called splicing.

<img src="https://github.com/CropEvol/lecture/blob/master/textbook_2022/images/splicing.png?raw=true" alt="splicing" height="250px" align="middle">

Because RNA-seq is sequencing expressed mRNA, only exon regions are sequenced.

Therefore, unlike mapping DNA sequences—where reads can simply be aligned to matching positions in the reference genome—**RNA-seq reads must be aligned across multiple exon regions that are separated and discontinuous**.

In other words, when mapping RNA-seq reads to a reference genome, intron gaps must be taken into account.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/RNA_mapping.png?raw=true" alt="RNA_mapping" height="160px" align="middle">

To handle this situation, mapping RNA-seq requires specialized tools designed for spliced alignment.

Tools such as [Hisat2](http://daehwankimlab.github.io/hisat2/) and [STAR](https://github.com/alexdobin/STAR) are suitable for RNA-seq mapping because they can account for intron gaps.

In this lecture, we will use [Hisat2](http://daehwankimlab.github.io/hisat2/).

Please open a terminal and install Hisat2 using the following command.

```
apt install hisat2
```

## Sample dataset

Now, let us proceed with a hands-on RNA-seq analysis using sample data of reference genome.

This sample reference genome is the one chromosome of rice blast fungus.

## Get reference genome data & RNA-seq data

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/mRNA_sequencing.png?raw=true" alt="central_dogma" height="250px" align="middle">

First, we will download the dataset.

The reference genome sequence of the rice blast fungus is available at:

* https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/assembly.fasta

The RNA-seq read data, obtained by RNA-seq from the rice blast fungus using a next-generation sequencer, is available at:

* https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/infect_0h_rep1.fastq.gz

This time, the data consist of single-end short reads.

Please open a terminal and download the two files above using the wget command.

```
wget URL_of_files
```

We will also install `seqkit`, a tool for handling sequence data.

```
apt install seqkit
```

If you examine the data using commands such as `less` or `seqkit`, you will see that the RNA-seq short reads used here are not very different from the DNA reads.

## Mapping obtained RNA-seq reads

Next, we will proceed to align the mRNA sequences obtained as short reads to the reference genome.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/mRNA_alignment.png?raw=true" alt="central_dogma" height="250px" align="middle">

As mentioned earlier, we will use `Hisat2`, a program specifically designed for RNA-seq mapping.


### Building index file for the reference genome

Before performing mapping, we first create an index file for the reference genome.

An index file is, as the name suggests, an index that allows the mapping tool to efficiently search the reference genome.

In general, an index file must be created separately for each tool.

You can create an index file for `Hisat2` using the following command:

```
hisat2-build <reference genome FASTA file> <index name>
```

Make sure to specify the correct FASTA file name and run the command.

If we set the index name to `genome_index`, the command will be:

```
hisat2-build assembly.fasta genome_index
```

If many files named `genome_index.*.ht2` are created, the index has been built successfully.


### Starting the Mapping

Once the index file has been created, we can proceed to map the RNA-seq reads.

Mapping can be performed using the hisat2 command.

```
hisat2 -x <index name> -U <short-read file> -S <output file>
```

In this case, the index name we created is `genome_index`, and the short-read file is `infect_0h_rep1.fastq.gz`, so the command becomes:

```
hisat2 -x genome_index -U infect_0h_rep1.fastq.gz -S infect_0h_rep1.sam
```
Here, the output file name is set to `infect_0h_rep1.sam`.

(This step should not take very long.)

Once the program finishes running, you should see a file named `infect_0h_rep1.sam`.

A SAM file contains the mapping results; in this case, it stores the RNA-seq mapping information as plain text.

SAM files tend to be large, so we will convert the file to BAM format and sort it by genomic position.

We will use a program called `samtools` to convert and sort the SAM file into BAM file.

Install `samtools` using the following command:

```
apt install samtools
```

After installing samtools, you can convert the SAM file to BAM format and sort it by adding the `-O bam` option to the `samtools sort` command:

```
samtools sort -O bam <SAM file> > <BAM file>
samtools index <BAM file>
```

Please run these commands on the created SAM file to create a BAM file named `infect_0h_rep1.bam`.

RNA-seq analyses are often performed across many different conditions (e.g., treatment A, treatment B, temperature C, temperature D, etc.) and then compared.

For this reason, RNA-seq data can become very large, and therefore, mapping results are generally stored in BAM format.


### Visualization of Mapping Results

At this point, we have obtained the results of mapping a large number of RNA-seq reads in the form of BAM (or SAM) files.

Then, we can visualize the mapping results using IGV.

By running the commands below, you can use IGV within Google Colab.

(IGV is typically used as a GUI application.)

```
pip install -q igv-notebook==0.3.1
wget -q -O modules.py https://raw.githubusercontent.com/slt666666/FAO_lecture/main/FAO_2024/data/modules.py
wget -q https://github.com/CropEvol/lecture/raw/master/textbook_2022/scripts/igv_prep.py -O igv_prep.py

samtools faidx assembly.fasta
```

Once the programs have been downloaded by commands above, IGV will start by running the following script.

In [ ]:
## Visualize alignment results
## import libraries
import igv_notebook
from modules import RefTrack, AnnotationTrack, BamTrack
igv_notebook.init()
# show reference genome
ref = RefTrack({ "fastaPath":"assembly.fasta", "indexPath":"assembly.fasta.fai", "id":"Strain_X" })
# show aligned reads
B = BamTrack({ "name":"infect_0h_rep1", "path":"infect_0h_rep1.bam", "indexPath":"infect_0h_rep1.bam.bai", "viewAsPairs":True })
# visualize
b = igv_notebook.Browser(ref)
b.load_track(B)


By visualizing the mapping results, you can see which regions of the genome are actively transcripted. You can also identify which parts of the genome correspond to exon regions and which correspond to intron regions.

In addition, you may notice some reads that span multiple exon regions. From this, it is possible to determine which exons are joined together to form a single mRNA—that is, which exons originate from the same gene.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/gene_annotation_RNAseq.png?raw=true" alt="gene_annotation_RNAseq" height="300px" align="middle">

## Gene Annotation

The process of identifying where genes are located in the genome, and determining which regions correspond to exons and which correspond to introns, is called **annotation**.

Since it is impractical to manually inspect all exon regions by eye, we use programs that extract information of regions of exon and intron from the BAM file containing the mapping results.

Well-known programs that can perform annotation based on RNA-seq data include [stringtie](https://ccb.jhu.edu/software/stringtie/) and [PsiCLASS](https://github.com/splicebox/PsiCLASS) ...etc.

Each program has its own characteristics; for example, some support long-read RNA-seq, while others are particularly good at detecting splicing variants.

In this exercise, we will use `stringtie`, so let’s install it as we have done with the other tools.

```
apt install stringtie
```

By using `stringtie`, we can perform gene annotation from an RNA-seq BAM file.

After annotation, a file called a GTF file is generated, which lists information such as the positions of exons.

GTF stands for Gene Feature Format.

The command is simple: you only need to specify the BAM file to use and the name of the output GTF file.

```
stringtie <BAM file> -o <output GTF file>
```

This time, let’s output the results to a file named `transcripts.gtf`.

If the command runs successfully, the file `transcripts.gtf` will be created.

If you examine the contents of this file using the `less -S` command, you will see entries that describe information such as which chromosome and which positions correspond to exon regions.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/stringtie_results.png?raw=true" alt="central_dogma" height="400px" align="middle">

By using IGV with a GTF file, you can add this gene annotation information to the mapping results you visualized earlier.

In [ ]:
## Visualize alignment results
## import libraries
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
igv_notebook.init()
## show reference info
ref = RefTrack({ "fastaPath":"assembly.fasta", "indexPath":"assembly.fasta.fai", "id":"Strain_X" })
## show alined reads info
A = BamTrack({ "name":"infect_0h_rep1", "path":"infect_0h_rep1.bam", "indexPath":"infect_0h_rep1.bam.bai", "viewAsPairs":True })
## visualize
b = igv_notebook.Browser(ref)
## show gtf information
b.load_track({"name": "Annotations", "type": "annotation", "format": "gtf", "displayMode": "EXPANDED", "path": "transcripts.gtf"})
b.load_track(A)

### Other Annotation Tools and Required Data

For gene annotation, there are also methods that do not rely on RNA-seq. These approaches use machine learning models or gene annotation information from other species to predict gene structures.

Examples include [Augustus](https://bioinf.uni-greifswald.de/augustus/) and [Helixer](https://github.com/weberlab-hhu/Helixer?tab=readme-ov-file) ...etc.

However, annotation results by these methods are just prediction. To accurately annotate genes that are **actually expressed**, RNA-seq data are essential.

For example, in plants, some genes are expressed only in leaves, others only in roots, and some only under heat stress or other specific conditions.

To annotate such genes, it is not sufficient to rely on RNA-seq data obtained from leaf samples alone. Instead, RNA-seq must be performed using samples from multiple tissues and under various conditions, and these datasets should be integrated to generate comprehensive gene annotations.

## Evaluate Gene Expression Levels

In DNA sequence mapping, the goal was to detect genetic variants.

In contrast, the main objective of RNA-seq is to obtain information on which genes are expressed and how strongly they are expressed.

*Note: It is also possible to detect genetic variants using RNA-seq.*

The reads obtained by RNA-seq depend on the extracted mRNA.

mRNAs from highly expressed genes are sequenced more frequently, whereas mRNAs from lowly expressed genes are sequenced less frequently.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/read_count.png?raw=true" alt="central_dogma" height="250px" align="middle">

Therefore, for each gene (or exon), regions with many mapped reads indicate high transcription levels, while regions with few mapped reads indicate low transcription levels.

Thus, by counting the number of reads mapped to each gene, we can quantify gene expression levels.

This process is also performed by the program, a well-known program for counting reads is `featureCounts`.

`featureCounts` is part of the `subread` tool suite, and it can be installed using the following command:

```
apt install subread
```

To obtain gene expression levels (read counts) from an RNA-seq BAM file using `featureCounts`, run the following command:

```
featureCounts -a <GTF file> -o <output file> <BAM file>
```

This time, let’s set the output file name to `counts.txt`.

The GTF file will be `transcripts.gtf`, which we created earlier, and the BAM file will be `infect_0h_rep1.bam`.

By examining the resulting `counts.txt` file, you can see how many reads are mapped to each gene.

This allows you to quantitatively assess which genes have high or low expression levels.




### Compare gene expression data from multiple samples

So far, we have covered a general method for examining gene expression levels.

However, in most research studies, gene expression is analyzed by comparing multiple samples.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/compare_RNAseq_en.png?raw=true" alt="central_dogma" height="250px" align="middle">

Next, we will look at how to handle gene expression data from multiple samples.

The RNA-seq data we used previpusly was from rice blast fungus under normal conditions.

As an additional sample, we will use RNA-seq data from rice blast fungus collected 48 hours after infection of a susceptible barley cultivar, Nigrate.

By comparing gene expression levels under normal conditions and during infection, we can identify genes that are required for infection.

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/mRNA_compare_en.png?raw=true" alt="central_dogma" height="250px" align="middle">

The RNA-seq data for the infected sample are available at:

* https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/infect_48h_rep1.fastq.gz

Please download the data using the `wget` command and use this RNA-seq short-read file (`infect_48h_rep1.fastq.gz`) to create a BAM file by following the same steps as before.

1. Mapping
```
hisat2 -x <index name> -U <short-read file> -S <output file>
```

2. Sorting and converting the mapping results to BAM format
```
samtools sort -O bam <SAM file> > <output>.bam
samtools index <BAM file>
```

If the files `infect_48h_rep1.bam` and `infect_48h_rep1.bam.bai` are created, the process was successful.

Next, we will compare gene expression levels between the normal condition and the infection condition.

`featureCounts` can handle multiple BAM files at once:

```
featureCounts -a <GTF file> -o <output file> <BAM file 1> <BAM file 2> <BAM file 3> ...
```

This time, specify two BAM files: `infect_0h_rep1.bam` and `infect_48h_rep1.bam`.

Set the output file name specified by `-o` to `counts.txt`.

When you view this file using a command such as `less -S`, you will see read counts for `infect_0h_rep1` and `infect_48h_rep1`.

These counts can then be compared.

However, in RNA-seq analysis, raw read counts cannot be directly compared between samples.

Read counts must be normalized because the amount of read is differed from sample to sample.

In addition, RNA-seq experiments are typically performed with at least three biological replicates per condition.

Because gene expression levels can be highly variable, replicates are necessary to obtain reliable results.

For example, to compare gene expression between non-infected and infected conditions, RNA-seq is usually performed three times under each condition, and the results are analyzed collectively.

We will use RNA-seq data from all six samples. We can download these dataset:

* Three replicates (non-infected)
  * https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/infect_0h_rep1.fastq.gz
  * https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/infect_0h_rep2.fastq.gz
  * https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/infect_0h_rep3.fastq.gz

* Three replicates (infected)
  * https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/infect_48h_rep1.fastq.gz
  * https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/infect_48h_rep2.fastq.gz
  * https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/infect_48h_rep3.fastq.gz

The same analysis steps as before will be applied to all six samples.

Since this involves repeating the same procedure multiple times, a shell script has been prepared to run all six analyses at once. Please execute the following commands:

```
wget https://raw.githubusercontent.com/slt666666/MOryzae_testdata/refs/heads/main/RNA_seq.sh

bash RNA_seq.sh
```
If a file named `counts_0h_vs_48h.txt` is created, the process was successful.

This means that read counts have been calculated for all six samples.

The gene annotation has also been generated using data from all samples.

Using these read count data with replicates, we can proceed to compare gene expression levels between samples.

## DEG Analysis

<img src="https://github.com/CropEvol/lecture/blob/master/images/exp_2025_2/DEG_en.png?raw=true" alt="central_dogma" height="400px" align="middle">

Genes whose expression levels differ greatly between samples are detected through an analysis called **DEG** (**Differentially Expressed Gene**) analysis.

Although DEG analysis can be performed by writing custom code from scratch, using R packages such as [edgeR](https://bioconductor.org/packages/release/bioc/vignettes/edgeR/inst/doc/edgeRUsersGuide.pdf) and [DEseq2](https://bioconductor.org/packages/release/bioc/manuals/DESeq2/man/DESeq2.pdf) makes it much easier to compare gene expression levels between samples, including proper normalization of read counts.

In this exercise, we will perform DEG analysis using [PyDESeq](https://pydeseq2.readthedocs.io), a program that allows DESeq2-based analysis to be run in Python.

Install `PyDESeq2` using the following command:

```
pip install pydeseq2
```

We will skip the detailed explanations here, but [the official tutorial](https://pydeseq2.readthedocs.io/en/stable/auto_examples/index.html) provides all the necessary code and commands.

By running the code below, you can identify differentially expressed genes (DEGs) between the infected condition (48 h) and the non-infected condition (0 h).

In [ ]:
import pandas as pd
from pydeseq2.utils import load_example_data
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

# organize dataset
counts_df = pd.read_csv("counts_0h_vs_48h.txt", sep="\t", comment="#").set_index("Geneid").iloc[:, 5:].T
metadata = pd.DataFrame(index=counts_df.index, data={"condition":["0h", "0h", "0h", "48h", "48h", "48h"]})

# remove genes with very low expression levels
genes_to_keep = counts_df.columns[counts_df.sum(axis=0) >= 10]
counts_df = counts_df[genes_to_keep]

# normalize read counts & DEG analysis
inference = DefaultInference()
dds = DeseqDataSet(
  counts=counts_df,
  metadata=metadata,
  design_factors="condition",
  refit_cooks=True,
  inference=inference,
)
dds.deseq2()
stat_res = DeseqStats(dds, contrast=["condition", "48h", "0h"], inference=inference)
stat_res.summary()

You can save the analysis results using the command below, and the results can be opened in Excel or similar software.

In [ ]:
# save the analysis result
stat_res.results_df.to_csv("DEG_0h_vs_48h.csv")

The obtained `log2FoldChange` indicates how large the difference in expression levels is between the compared conditions.

A `log2FoldChange` of 1 corresponds to a twofold difference in expression, while a value of 2 corresponds to a fourfold difference.

The value `padj` represents the adjusted p-value.

Because the results include all genes, it is necessary to extract those that meet the criteria for being considered “differentially expressed.”

A common criterion is to define genes as DEGs when the absolute value of `log2FoldChange` is ≥ 1 and the adjusted p-value (`padj`) is < 0.05.

In [ ]:
# Display the top 10 genes with the strongest expression changes
stat_res.results_df.sort_values(by=["padj"]).head(10)

By examining the read coverage in IGV, you can clearly see the differences in expression levels.

In [ ]:
## show RNA-seq Alignment Results
## library preparation
import igv_notebook
from igv_prep import RefTrack, AnnotationTrack, BamTrack
igv_notebook.init()
## show reference genome
ref = RefTrack({ "fastaPath":"assembly.fasta", "indexPath":"assembly.fasta.fai", "id":"Strain_X" })
## show alignment results
A = BamTrack({ "name":"infect_0h_rep1", "path":"infect_0h_rep1.bam", "indexPath":"infect_0h_rep1.bam.bai", "viewAsPairs":True })
B = BamTrack({ "name":"infect_48h_rep1", "path":"infect_48h_rep1.bam", "indexPath":"infect_48h_rep1.bam.bai", "viewAsPairs":True })
## visualize
b = igv_notebook.Browser(ref)
## show gene annotation
b.load_track({"name": "Annotations", "type": "annotation", "format": "gtf", "displayMode": "EXPANDED", "path": "transcripts.gtf"})
b.load_track(A)
b.load_track(B)

### Investigating strongly differentially expressed genes using BLAST

From this analysis, you can see that the gene `STRG.1697` shows a sharp decrease in expression during infection.

In contrast, the gene `STRG.1269` shows a dramatic increase in expression during infection.

To investigate the functions of these genes, we first need to extract their sequences.

Using a tool called `gffread`, you can extract gene sequences based on the GTF gene annotation file and the genome sequence created earlier.

```
apt install gffread
```

The following command extracts the sequences of all exon regions:

```
gffread transcripts.gtf -g assembly.fasta -w transcripts.fasta
```

The resulting file, transcripts.fasta, contains the sequences of all genes.

Next, to find specific genes such as `STRG.1697` or `STRG.1269`, you can use the following seqkit command.

*Note: Please add “.1” to the end of the gene name you want to search for.*

*Example: to search for `STRG.1697`, use `STRG.1697.1`.*

```
seqkit grep -p <gene_name> transcripts.fasta > <output_file>
```

This allows you to extract sequences using the gene name as a key.

You can then submit the extracted sequences to [BLAST](https://blast.ncbi.nlm.nih.gov/Blast.cgi) to investigate the potential functions of genes showing large expression changes.



### Visualization of DEG Results: Volcano Plot

Results from DEG analysis are often visualized using a **volcano plot**.

In this exercise, we write code in Python to generate the plot, but in R there are packages that can automatically generate volcano plots.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

df = stat_res.results_df
df['nlog10'] = -np.log10(df.padj)

def dot_color(a):
    logFC, nlog10 = a

    if abs(logFC) <= 1 or nlog10 <= 2:
        return 'Not significant'
    if logFC > 1 and nlog10 >2:
        return 'Up'
    if logFC < -1 and nlog10 >2:
        return 'Down'
df['color'] = df[['log2FoldChange', 'nlog10']].apply(dot_color, axis = 1)

plt.figure(figsize = (7,7))

ax = sns.scatterplot(data = df, x = 'log2FoldChange', y = 'nlog10',
                    hue = 'color', hue_order = ['Not significant', 'Up', 'Down'],
                    palette = ['dimgrey', 'firebrick', 'darkslateblue'])

ax.axhline(2, zorder = 0, c = 'k', lw = 2, ls = '--')
ax.axvline(1, zorder = 0, c = 'k', lw = 2, ls = '--')
ax.axvline(-1, zorder = 0, c = 'k', lw = 2, ls = '--')

for axis in ['bottom', 'left']:
    ax.spines[axis].set_linewidth(2)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.tick_params(width = 2)

plt.xticks(size = 12, weight = 'bold')
plt.yticks(size = 12, weight = 'bold')

plt.xlabel("$log_{2}$ fold change", size = 15)
plt.ylabel("-$log_{10}$ q-value", size = 15)

plt.show()